## Store Traces in Unity Catalog Tables

### Installing Utilities and Libraries

In [ ]:
%pip install -qq --upgrade "mlflow[databricks]>=3.14.0" openai databricks-sdk==0.77.0

### Restarting the Python Environment

In [ ]:
dbutils.library.restartPython()

### Create the Tracing Schema in Unity Catalog

In [ ]:
%sql

CREATE SCHEMA IF NOT EXISTS mlflow_tracing.traces;

### Create an Experiment with a Unity Catalog Trace Location

In [ ]:
import os
import mlflow
from mlflow.entities.trace_location import UnityCatalog

mlflow.set_tracking_uri("databricks")

# Specify the ID of a SQL warehouse you have access to.
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = "<SQL_WAREHOUSE_ID>"
# Specify the name of the MLflow Experiment to use for viewing traces in the UI.
experiment_name = "/Shared/demo-uc-tracing-experiment"
# Specify the name of the Catalog to use for storing traces.
catalog_name = "mlflow_tracing"
# Specify the name of the Schema to use for storing traces.
schema_name = "traces"
# Specify the name of the prefix appended to every table storing trace data.
table_prefix = "demouctracing"

In [ ]:
# mlflow.set_experiment is an upsert operation
experiment = mlflow.set_experiment(
    experiment_name=experiment_name,
    trace_location=UnityCatalog(
        catalog_name=catalog_name,
        schema_name=schema_name,
        table_prefix=table_prefix,  # defaults to experiment id if not provided
    ),
)

print(f"Experiment ID: {experiment.experiment_id}")
print(experiment.trace_location.full_otel_spans_table_name)

### Define the Application Function

In [ ]:
from openai import OpenAI

# Use the trace decorator to capture the application's entry point
@mlflow.trace
def my_app(input: str):
    # create the OpenAI Client
    client = OpenAI(
        api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
        base_url = "YOUR_DATABRICKS_WORKSPACE_HOSTNAME/serving-endpoints"
    )
    # This call is automatically instrumented by `mlflow.openai.autolog()`
    # OpenAI Request
    completion = client.chat.completions.create(
        model="databricks-claude-sonnet-4-5",
        messages=[
            {
                "role":"system",
                "content": [
                    {
                        "type": "text", "text": "You are Batman, the protector of Gotham City"
                    }
                ]

            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text", "text": input
                    }
                ]
            }
        ]
    )

    # printing the response
    return completion.choices[0].message.content

### Trace Multiple Application Runs

In [ ]:
queries = [
    "Hi, My name is Kuljot!",
    "How is Gotham City doing today?",
    "Any crimes worrying Gotham?",
    "What is the Joker upto now?"
]

for query in queries:
    print("Query: {}".format(query))
    print("\n\n")
    response = my_app(query)
    print("Assistant Response: {}".format(response))
    print("\n\n")
    print("=================================================")
    print("\n\n")